In [ ]:
pre_backward()  # 等待 chunk0 的梯度到达（来自下游stage）
    
# chunk0 反向传播 micro-batch 0
backward_input_tensor_grad = backward_step_helper(0)

post_backward(backward_input_tensor_grad)  # 发送梯度给上游

# pre_backward

- 核心目的
  - 确保反向传播所需的梯度张量已从下游 stage 接收完毕，才能开始当前 chunk 的反向计算。

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              Steady 1F1B 阶段                                │
├─────────────────────────────────────────────────────────────┤
│  前向: pre_forward → forward_step → post_forward            │
│       ↑等待输入      ↑计算      ↑发送输出+预接收              │
├─────────────────────────────────────────────────────────────┤
│  反向: pre_backward → backward_step → post_backward         │
│       ↑本代码所在位置  ↑计算      ↑发送梯度+预接收              │
└─────────────────────────────────────────────────────────────┘

## 为什么需要等待接收梯度？
### 反向传播链

In [ ]:
PP 拓扑（2个stage）:
Stage 0 (GPU 0): chunk0, chunk2  ← 靠近输入端
Stage 1 (GPU 1): chunk1, chunk3  ← 靠近输出端，计算loss

<b>反向传播方向:</b> <font color='red'>从输出端往输入端传播</font>

反向传播:
  GPU 1 chunk3 反向 → 输入梯度 → 给 GPU 0 chunk2 反向
  GPU 1 chunk1 反向 → 输入梯度 → 给 GPU 0 chunk0 反向

### 具体到 VPP

In [ ]:
GPU 1 (Stage 1, VPP=2):
  v_id=0: chunk3 反向 (mb=0) → 产生梯度 → 发送给 GPU 0 的 chunk2
  
GPU 0 (Stage 0, VPP=2):
  要执行 chunk2 反向 (mb=0) → 必须先收到来自 GPU 1 的梯度！
  
  pp_pre_backward() 就是做这个等待的！

### 反向传播流程（Steady 1F1B）

In [ ]:
假设当前迭代: backward_k=0

1. 前置状态（来自上次 post_backward）:
   recv_next_wait_handles = [handle_from_GPU1_chunk3]
   ← 上次反向时预接收的梯度

2. 进入 pp_pre_backward():
   
   a. vp_stage = get_model_chunk_id(0, forward=False) → 1 (chunk3)
      反向表: v_id=0 → ch=1
      
   b. _is_vp_last_stage(1)? chunk3 是 VPP尾 → True
      is_pp_last_stage(pp_group)? GPU 0 是 Stage 0 → False
      True and False = False → 不是真正的末阶段！
      → 需要接收梯度！
   
   c. config.overlap_p2p_comm_warmup_flush = True
      → 走断言分支
   
   d. assert recv_next_wait_handles 非空 ✓
      → pop(0) 得 handle_from_GPU1_chunk3
      → handle.wait() → 阻塞等待梯度到达
   
   e. 梯度到达！继续执行 backward_step_helper()

3. 对比：如果是 GPU 1 (Stage 1, PP末阶段)
   
   a. vp_stage = get_model_chunk_id(0, forward=False) → 1 (chunk3)
   
   b. _is_vp_last_stage(1)? True
      is_pp_last_stage(pp_group)? GPU 1 是 Stage 1 → True
      True and True = True → 是真正的末阶段！
      → 条件 not(...) = False → 跳过等待！
   
   c. 直接执行 backward_step_helper()
      loss.backward() → 自己产生梯度，不需要接收

### 与 pp_post_backward 的配合

In [ ]:
迭代 k:
  pre_backward():
    wait(recv_next_wait_handles[0])  ← 等 post_backward_{k-1} 预接收的梯度
    ← 本代码
  
  backward_step():
    compute → input_tensor_grad（传给上游的梯度）
  
  post_backward():
    send(input_tensor_grad)            ← 发起异步发送给上游
    recv_next_wait_handles.append(...)  ← 预接收下次梯度

迭代 k+1:
  pre_backward():
    wait(recv_next_wait_handles[0])  ← 等 post_backward_k 预接收的梯度
    ...

In [ ]:
            '''反向预处理'''
            # Sync backward recv
            def pp_pre_backward(vp_stage=None):
                
                '''
                缓存的"接收下游梯度"异步句柄列表
                
                为什么是 next（下游）？
                
                反向传播方向: 从输出端往输入端
                  "下游" = 靠近输出端的方向 = 梯度的来源方向
                  
                  GPU 0 的 chunk2 反向 ← 需要 GPU 1 的 chunk3 反向梯度
                  ← 来源是"下一个"stage（next stage）
                '''
                nonlocal recv_next_wait_handles
                '''
                确定当前虚拟阶段:
                
                forward=False 的关键作用：反向时查反向调度表
                '''
                if vp_stage is None:
                    vp_stage = get_model_chunk_id(backward_k, forward=False)
                    
                '''等待接收反向梯度（来自下游 stage）'''
                '''
                末阶段短路判断
                | 条件                           | 含义                             |
                | ----------------------------- | ------------------------         |
                | `_is_vp_last_stage(vp_stage)` | 当前 chunk 是 VPP 的最后一个虚拟阶段 |
                | `is_pp_last_stage(pp_group)`  | 当前 GPU 是 PP 的最后一个物理阶段    |
                | `and`                         | **同时满足 = 真正的输出端，梯度源头**   |
                
                为什么末阶段不需要等待接收梯度？
                
                PP 末阶段（GPU 1）:
                  - 直接计算 loss
                  - loss.backward() 产生梯度
                  - 是梯度的"源头"，不需要从别人那里接收
  
                PP 中间/首阶段（GPU 0）:
                  - 需要下游 stage 传回来的梯度
                  - 必须等待 recv_next 完成！
                '''
                if not (_is_vp_last_stage(vp_stage=vp_stage) and is_pp_last_stage(pp_group)):
                    '''
                    等待接收句柄（warmup_flush 模式）
                    
                    - 特点：Warmup 阶段就开始预取梯度，Flush 机制确保顺序
                    '''
                    if config.overlap_p2p_comm_warmup_flush:
                        '''
                        保护场景：如果列表为空，说明前置的 post_forward 或 post_backward 没有正确发起预接收
                        '''
                        assert recv_next_wait_handles, (
                            f'哪个 GPU 出错:pp rank {pipeline_parallel_rank}, 反向第几次迭代:bwd iteration {backward_k}, '
                            'should have registered recv next handle'
                        )
                        '''
                        弹出并等待
                        
                        为什么是 pop(0)（FIFO）？
                        
                        时间线:
                        t0: post_backward() 发起预接收 mb=0 的梯度 → handle_A
                        t1: post_backward() 发起预接收 mb=1 的梯度 → handle_B
                        t2: pre_backward() 需要 mb=0 的梯度 → pop(0) 得 handle_A ✓
                        t3: pre_backward() 需要 mb=1 的梯度 → pop(0) 得 handle_B ✓

                        如果 pop()（LIFO）:
                        t2: pop() 得 handle_B → 但需要的是 mb=0！✗ 顺序错乱
                        '''
                        recv_next_wait_handle = recv_next_wait_handles.pop(0)# FIFO 队列
                        recv_next_wait_handle.wait()                         # 阻塞等待完成
                    else:
                        '''等待接收句柄（常规重叠模式）'''
                        '''
                        与 warmup_flush 模式的区别：
                        
                        |             | `warmup_flush`              | 常规 `overlap_p2p_comm` |
                        | ------      | --------------              | --------------------- |
                        | 断言检查     | 严格，必须非空             | 宽松，允许为空               |
                        | 预取启动时机  | Warmup 阶段就开始         | Steady 阶段才开始          |
                        | 适用场景     | 长流水线，需要更早隐藏延迟  | 短流水线，标准 1F1B          |

                        '''
                        if recv_next_wait_handles is not None and recv_next_wait_handles:
                            
                            recv_next_wait_handle = recv_next_wait_handles.pop(0) # FIFO 队列
                            recv_next_wait_handle.wait()                          # 阻塞等待完成


# post_backward

In [ ]:

            '''反向后处理'''
            # Async backward send / receive
            def pp_post_backward(input_tensor_grad, vp_stage=None):
                nonlocal send_prev_wait_handle
                nonlocal bwd_wait_handles
                nonlocal recv_next_wait_handles
                if vp_stage is None:
                    vp_stage = get_model_chunk_id(backward_k, forward=False)
                '''1. 首阶段截断（无梯度可发送）'''
                # First virtual stage no activation gradient tensor to send.
                if _is_vp_first_stage(vp_stage=vp_stage) and is_pp_first_stage(pp_group):
                    input_tensor_grad = None

                recv_next, next_backward_model_chunk_id = recv_tensor_from_previous_stage(
                    backward_k, forward=False
                )

                '''2. 发起异步 P2P（发送梯度 + 预接收下一个梯度）'''
                (bwd_recv_buffer[backward_k % bwd_recv_buffer_size], bwd_wait_handles) = (
                    p2p_communicator.send_backward_recv_backward(
                        input_tensor_grad,
                        recv_next=recv_next,
                        tensor_shape=tensor_shape,
                        overlap_p2p_comm=True,
                    )
                )
                if send_prev_wait_handle is not None:
                    send_prev_wait_handle.wait()
                if bwd_wait_handles is not None:
                    send_prev_wait_handle = (
                        bwd_wait_handles.pop("send_prev")
                        if "send_prev" in bwd_wait_handles
                        else None
                    )
                    if "recv_next" in bwd_wait_handles:
                        recv_next_wait_handles.append(bwd_wait_handles.pop("recv_next"))

                # Put input_tensor and output_tensor_grad in data structures in the
                # right location.

                '''3. 缓存接收的梯度'''
                if recv_next:
                    output_tensor_grads[next_backward_model_chunk_id].append(
                        bwd_recv_buffer[backward_k % bwd_recv_buffer_size]
                    )
                    bwd_recv_buffer[(backward_k + 1) % bwd_recv_buffer_size] = None
                return input_tensor_grad


# backward_step_helper
- backward_step_helper 是 VPP 调度中的"反向翻译官+执行器"：
  - 通过反向调度表将虚拟执行序号映射为具体的模型 chunk 和数据批次，
  - 取出前向缓存的激活值和下游传来的梯度，执行 PyTorch 的链式法则反向传播，
  - 清理已用完的缓存释放内存，最终返回输入梯度供 post_backward 发送给上游 stage——
  - 完成"接收梯度→计算梯度→发送梯度"的完整反向链路。
- 这段代码是 Megatron-LM 中 VPP（虚拟流水线并行）的反向传播核心辅助函数
- <font color='red'>负责将虚拟微批次映射到具体的模型 chunk，执行实际的反向计算。</font>

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              backward_step_helper                            │
│              （虚拟微批次 → 真实反向计算）                     │
├─────────────────────────────────────────────────────────────┤
│  1. 查表：v_id → model_chunk_id（反向调度表）                 │
│  2. 预处理：准备输入、输出、梯度张量                          │
│  3. 计算：调用 backward_step 执行真实反向                     │
│  4. 后处理：清理缓存状态                                      │
│  5. 返回：输入梯度（传给上游 stage）                          │
└─────────────────────────────────────────────────────────────┘

In [ ]:
配置:
  model = [chunk0 (layers 0-3), chunk1 (layers 4-7)]
  input_tensors = [[mb0_in, mb1_in, ...], [mb0_in, mb1_in, ...]]
  output_tensors = [[mb0_out, ...], [mb0_out, ...]]
  output_tensor_grads = [[mb0_grad, ...], [mb0_grad, ...]]

执行 backward_step_helper(3):

Step 1: 查表
  model_chunk_id = get_model_chunk_id(3, forward=False) → 0
  反向表: v_id=3 → ch=0

Step 2: 预处理
  input_tensor, output_tensor, output_tensor_grad = backward_step_helper_preprocess(3, 0)
  
  microbatch_id = get_microbatch_id_in_model_chunk(3, forward=False) → 0
  
  input_tensor = input_tensors[0][0]      # mb0 的 chunk0 输入
  output_tensor = output_tensors[0][0]    # mb0 的 chunk0 输出
  output_tensor_grad = output_tensor_grads[0][0]  # 下游传来的梯度

Step 3: 计算
  input_tensor_grad = backward_step(
      input_tensor,      # mb0 输入 chunk0 的激活值
      output_tensor,     # mb0 经过 chunk0 的输出
      output_tensor_grad, # 下游（chunk1）传来的梯度
      config,
  )
  
  内部执行:
    # 链式法则: d(loss)/d(input) = output_tensor_grad × d(output)/d(input)
    output_tensor.backward(gradient=output_tensor_grad)
    input_tensor_grad = input_tensor.grad

Step 4: 后处理
  backward_step_helper_postprocess(3)
  
  # 清理 mb0 的缓存（已完成反向，不再需要）
  input_tensors[0].pop(0)
  output_tensors[0].pop(0)
  output_tensor_grads[0].pop(0)

Step 5: 返回
  return input_tensor_grad  # 传给 post_backward 发送给上游

In [ ]:

    def backward_step_helper(virtual_microbatch_id):
        """Helper method to run backward step with model split into chunks"""
        '''
        需要读取和修改外层的 output_tensor_grads
        output_tensor_grads: 缓存的梯度张量列表，每个 chunk 一个子列表
        output_tensor_grads = [[], []]  # VPP=2
        '''
        nonlocal output_tensor_grads
        
        '''
        查表获取模型 Chunk ID
        
        关键：forward=False 使用反向调度表
        '''
        model_chunk_id = get_model_chunk_id(virtual_microbatch_id, forward=False)

        '''
        预处理：准备反向传播所需张量
        
        | 返回值                  | 来源                                    | 说明             |
| -------------------- | ------------------------------------- | -------------- |
| `input_tensor`       | `input_tensors[model_chunk_id]`       | 前向时保存的输入激活值    |
| `output_tensor`      | `output_tensors[model_chunk_id]`      | 前向时保存的输出激活值    |
| `output_tensor_grad` | `output_tensor_grads[model_chunk_id]` | 下游 stage 传来的梯度 |

        '''
        input_tensor, output_tensor, output_tensor_grad = backward_step_helper_preprocess(
            virtual_microbatch_id, model_chunk_id
        )

        '''
        执行反向计算
        '''
        input_tensor_grad = backward_step(input_tensor, output_tensor, output_tensor_grad, config)

        '''
        后处理
        
        | 操作            | 说明                                                           |
| ------------- | ------------------------------------------------------------ |
| 释放前向缓存的激活值    | `input_tensors[chunk].pop(0)`，`output_tensors[chunk].pop(0)` |
| 释放梯度张量        | `output_tensor_grads[chunk].pop(0)`                          |
| 标记 chunk 为已同步 | 如果配置了梯度同步                                                    |
| 内存优化          | `torch.cuda.empty_cache()`（可选）                               |

        为什么需要清理？
        VPP 交错调度，内存紧张:
          input_tensors[0] = [tensor_mb0, tensor_mb1, tensor_mb2, ...]
  
          mb0 的反向完成后，mb0 的输入不再需要
          → pop(0) 释放，避免无限累积
        '''
        backward_step_helper_postprocess(virtual_microbatch_id)

        return input_tensor_grad


## 返回输入梯度 input_tensor_grad
### 返回值的用途：

In [ ]:
input_tensor_grad 被 post_backward 用于:
  1. 发送给上游 stage（作为它的 output_tensor_grad）
  2. 或如果是首阶段，置为 None（无需发送）

pp_post_backward(input_tensor_grad):
  if 不是首阶段:
      send(input_tensor_grad)  # P2P 发送给 prev stage
      recv_next = True         # 预接收下次梯度
  else:
      input_tensor_grad = None # 首阶段，无上游可传

## 与 forward_step_helper 的对比

|           | `forward_step_helper`    | `backward_step_helper`     |
| --------- | ------------------------ | -------------------------- |
| **查表方向**  | `forward=True`           | `forward=False`            |
| **输入来源**  | `data_iterator` 或 P2P 接收 | 前向缓存的激活值                   |
| **额外输入**  | 无                        | `output_tensor_grad`（下游梯度） |
| **计算内容**  | 模型前向                     | `backward()` 链式法则          |
| **返回值**   | `output_tensor`（激活值）     | `input_tensor_grad`（输入梯度）  |
| **后处理**   | 保存输出                     | 清理缓存、释放内存                  |
| **返回值用途** | 发送给下游 stage              | 发送给上游 stage                |


## 与外层调用的关系


In [ ]:
外层 Steady 循环:
  for k in range(num_microbatches_remaining):
      backward_k = k  # 如 0
      
      # 1. 反向预处理（等待梯度）
      pp_pre_backward()
      
      # 2. 执行反向计算（本函数）
      input_tensor_grad = backward_step_helper(backward_k)
      
      # 3. 反向后处理（发送梯度）
      input_tensor_grad = pp_post_backward(input_tensor_grad)

## backward_step_helper_preprocess
- 这段代码是 Megatron-LM 中 VPP（虚拟流水线并行）反向传播前的预处理函数，
- 负责梯度同步控制、末阶段特殊处理和缓存张量提取
backward_step_helper_preprocess 是反向传播前的"三合一准备站"：
- 第一，在最后一个微批次时退出 no_sync 模式<font color='red'>触发 DDP 梯度同步</font>；
- 第二，为 PP 末阶段自动注入 None 梯度（loss 是梯度源头无需接收）；
- 第三，从 FIFO 队列中按序弹出前向缓存的输入、输出和下游梯度——确保 backward_step 有完整的计算图和梯度链可用。

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│         backward_step_helper_preprocess                      │
│              （反向传播预处理）                               │
├─────────────────────────────────────────────────────────────┤
│  1. 梯度同步控制（默认 DDP 路径）                             │
│  2. 末阶段梯度初始化（PP 末端无下游梯度）                     │
│  3. 提取前向缓存的三类张量                                   │
│     - 输入激活值                                             │
│     - 输出激活值                                             │
│     - 下游梯度                                               │
└─────────────────────────────────────────────────────────────┘

In [ ]:
配置:
  model = [chunk0, chunk1, chunk2, chunk3]  # 分布在两个GPU
  GPU 0: chunk0, chunk2
  GPU 1: chunk1, chunk3  ← PP末阶段

执行 backward_step_helper_preprocess(0, 1):
  v_id=0, model_chunk_id=1 (chunk1)

Step 1: 梯度同步检查
  config.grad_sync_func is None? → True（使用默认DDP）
  
  is_last_microbatch_for_model_chunk(0)?
    查反向调度表: v_id=0 → mb=0, ch=1
    chunk1 的所有 mb: 0, 1, 2, 3, 4
    v_id 对应 chunk1 的: 0, 1, 2, 6, 7
    最后一个 v_id 是 7，不是 0
    → False！不启用同步
  
  （实际上应该在 v_id=7 时才同步）

Step 2: 末阶段检查
  _is_vp_last_stage(1)? chunk1 不是 VPP尾（VPP=2时尾是1? 需要确认）
  
  假设 VPP=2, chunk 编号 0,1:
    _is_vp_last_stage(1)? True（chunk1 是最后一个）
    is_pp_last_stage(pp_group)? GPU 1 是 PP末 → True
    → 条件满足
    
    output_tensor_grads[1] = []? 
      如果是首次反向 chunk1 → 空列表
      len([]) == 0 → True
      → append(None)
      
    output_tensor_grads[1] = [None]

Step 3: 提取张量
  input_tensor = input_tensors[1].pop(0)      # mb0 的 chunk1 输入
  output_tensor = output_tensors[1].pop(0)    # mb0 的 chunk1 输出
  output_tensor_grad = output_tensor_grads[1].pop(0)  # → None
  
  return input_tensor, output_tensor, None

In [ ]:
    def backward_step_helper_preprocess(virtual_microbatch_id, model_chunk_id):
        """Preprocess for backward_step_helper"""
        # launch grad synchronization (default)
        '''
        梯度同步控制:
        
        config.grad_sync_func is None	未配置自定义梯度同步函数（使用默认 DDP 行为）
        is_last_microbatch_for_model_chunk(v_id)	当前是该 chunk 的最后一个微批次
        '''
        if config.grad_sync_func is None and is_last_microbatch_for_model_chunk(
            virtual_microbatch_id
        ):
            '''
            为什么只在最后一个微批次启用同步？
            
            VPP 交错调度，同一 chunk 处理多个 micro-batch:
              chunk0: mb0, mb1, mb2, mb3, mb4
  
              每个 mb 反向传播后，梯度累加到 .grad
  
              如果每个 mb 都同步：
                → 频繁的 All-Reduce，通信开销大
                → 且梯度不完整（只累加了部分 mb）
  
              只在最后一个 mb 同步：
                → 所有 mb 的梯度都已累加
                → 一次 All-Reduce 得到完整梯度
                → 效率高！
            '''
            enable_grad_sync() # 退出 no_sync 模式
            synchronized_model_chunks.add(model_chunk_id) # 记录哪些 chunk 已完成同步

        '''
        末阶段梯度初始化
        
        为什么末阶段需要特殊处理？
        
        PP 末阶段（靠近 loss）:
          - 直接计算 loss
          - loss.backward() 不需要上游传来的梯度
          - 自己是梯度的"源头"

        但 backward_step 需要 output_tensor_grad 参数:
          torch.autograd.backward(output_tensor, grad_tensors=output_tensor_grad)
  
          如果 output_tensor_grad is None:
            PyTorch 默认使用 torch.ones_like(output_tensor)
            → 等价于 loss.backward()
        '''
        # pylint: disable=E0606
        if _is_vp_last_stage(vp_stage=model_chunk_id) and is_pp_last_stage(pp_group):
            if len(output_tensor_grads[model_chunk_id]) == 0:
                output_tensor_grads[model_chunk_id].append(None)
                
        input_tensor = input_tensors[model_chunk_id].pop(0)
        output_tensor = output_tensors[model_chunk_id].pop(0)
        output_tensor_grad = output_tensor_grads[model_chunk_id].pop(0)

        '''
         Warmup + Steady 阶段填充:
         input_tensors[0] = [mb0_in, mb1_in, mb2_in, ...]  ← FIFO 队列
         output_tensors[0] = [mb0_out, mb1_out, ...]
         output_tensor_grads[0] = [mb0_grad, mb1_grad, ...]  ← 反向时填充
        '''
        return input_tensor, output_tensor, output_tensor_grad


## backward_step_helper_postprocess
- 这段代码是 Megatron-LM 中 VPP（虚拟流水线并行）反向传播后的后处理函数
- <font color='red'>负责自定义梯度同步的触发时机和禁用梯度同步</font>
- 是反向传播后的"错峰同步调度器"：
  - 通过 v_id - pipeline_parallel_rank 的偏移计算，
  - 让不同 PP rank 在不同虚拟微批次检查不同 chunk 的完成状态，分散触发自定义梯度同步的时间点，避免所有 rank 同时通信造成的空闲等待——
  - 最后无论是否同步都重新禁用梯度同步，为下一个 micro-batch 的反向累加做准备。

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│         backward_step_helper_postprocess                     │
│              （反向传播后处理）                               │
├─────────────────────────────────────────────────────────────┤
│  1. 自定义梯度同步（config.grad_sync_func 非空）            │
│     - 计算同步的虚拟微批次 ID（考虑 PP rank 偏移）            │
│     - 判断是否是该 chunk 的最后一个微批次                     │
│     - 启用梯度同步并执行自定义同步函数                        │
│  2. 禁用梯度同步（为下一个 micro-batch 做准备）               │
│     - 重新进入 no_sync 模式                                 │
└─────────────────────────────────────────────────────────────┘

### 核心设计思想：为什么同步时机要考虑 pipeline_parallel_rank？
#### 问题背景：不同 PP stage 的计算时间差异

In [ ]:
PP=4 时，不同 stage 的反向传播时间可能不同：
  Stage 0 (GPU 0): 靠近输入，层数可能不同
  Stage 1 (GPU 1): ...
  Stage 2 (GPU 2): ...
  Stage 3 (GPU 3): 靠近输出，计算 loss

如果所有 stage 同时触发 grad_sync：
  → 快的 stage 等慢的 stage → 空闲气泡
  
解决方案：错峰触发，让通信与计算重叠

#### 错峰策略

In [ ]:
PP rank 0: 在 v_id=0 时同步（如果满足条件）
PP rank 1: 在 v_id=1 时同步（延迟1步）
PP rank 2: 在 v_id=2 时同步（延迟2步）
PP rank 3: 在 v_id=3 时同步（延迟3步）

通过 pipeline_parallel_rank 偏移实现错峰

#### 为什么用 grad_sync_chunk_id 而不是 model_chunk_id？

In [ ]:
因为错峰策略，当前 v_id 可能对应同步另一个 chunk 的梯度

示例：
  当前在反向 chunk 0
  但计算出的 grad_sync_chunk_id = 1
  → 同步 chunk 1 的梯度（它之前已完成所有 micro-batch 的反向）

In [ ]:
假设：config.grad_sync_func = [sync_func_0, sync_func_1]

执行 backward_step_helper_postprocess(v_id=3):

Step 1: 检查条件
  config.grad_sync_func is not None → True

Step 2: 计算偏移 v_id
  grad_sync_v_id = 3 - 0 = 3

Step 3: 检查有效性
  3 >= 0 → True

Step 4: 检查是否是最后一个 mb
  is_last_microbatch_for_model_chunk(3)?
  
  反向调度表:
  v_id | 0  1  2  3  4  5  6  7  8  9
  mb   | 0  1  2  0  1  2  3  4  3  4
  ch   | 1  1  1  0  0  0  1  1  0  0
  
  v_id=3 → ch=0, mb=0
  chunk 0 的所有 mb: 0,1,2,3,4（对应 v_id 3,4,5,8,9）
  最后一个 v_id 是 9，不是 3
  → False！不执行同步

执行 backward_step_helper_postprocess(v_id=9):

Step 2: grad_sync_v_id = 9 - 0 = 9
Step 3: 9 >= 0 → True
Step 4: is_last_microbatch_for_model_chunk(9)?
  v_id=9 → ch=0, mb=4
  chunk 0 的最后一个 mb 是 4（v_id=9）
  → True！执行同步

Step 5: 获取 chunk ID
  grad_sync_chunk_id = get_model_chunk_id(9, forward=False) → 0

Step 6: 执行同步
  enable_grad_sync()  # 退出 no_sync
  config.grad_sync_func[0](model[0].parameters())  # 同步 chunk 0
  synchronized_model_chunks.add(0)

Step 7: 禁用同步
  disable_grad_sync()  # 重新进入 no_sync

In [ ]:

    def backward_step_helper_postprocess(virtual_microbatch_id):
        """Postprocess for backward_step_helper"""
        # launch grad synchronization (custom grad sync)
        # Note: Asynchronous communication tends to slow down compute.
        # To reduce idling from mismatched microbatch times, we launch
        # asynchronous communication at the same time across the
        # pipeline-parallel group.
        '''
        自定义梯度同步条件
        
        | 场景     | `grad_sync_func`          | 处理方式                                    |
        | ------ | ------------------------- | --------------------------------------- |
        | 默认 DDP | `None`                    | 在 `backward_step_helper_preprocess` 中处理 |
        | 自定义同步  | 非空（如 DeepSpeed ZeRO、FSDP） | 在本函数中处理                                 |


        为什么分两个地方处理？
        
        preprocess 中的同步（默认 DDP）：
          - 简单，直接 enable_grad_sync()
          - 不依赖复杂的 rank 偏移计算

        postprocess 中的同步（自定义）：
          - 需要精确控制同步时机
          - 考虑 PP rank 差异，实现错峰
          - 避免所有 rank 同时触发通信
        '''
        if config.grad_sync_func is not None:
            '''
            计算同步的虚拟微批次 ID:
            
            为什么要减去 pipeline_parallel_rank？
            
            示例：PP=4，当前 v_id=3

            GPU 0 (rank=0): 3 - 0 = 3
            GPU 1 (rank=1): 3 - 1 = 2
            GPU 2 (rank=2): 3 - 2 = 1
            GPU 3 (rank=3): 3 - 3 = 0

            判断条件：grad_sync_v_id >= 0

            GPU 0: 3 >= 0 → True，检查 is_last_microbatch_for_model_chunk(3)
            GPU 1: 2 >= 0 → True，检查 is_last_microbatch_for_model_chunk(2)
            GPU 2: 1 >= 0 → True，检查 is_last_microbatch_for_model_chunk(1)
            GPU 3: 0 >= 0 → True，检查 is_last_microbatch_for_model_chunk(0)

            结果：不同 GPU 在不同 v_id 检查不同的"最后一个 micro-batch"
            
            
            错峰效果：
            
            时间线（v_id 递增）:

            v_id=0:
              GPU 0: 0-0=0, 检查 mb0 是否是 chunk 末 → 可能同步
              GPU 1: 0-1=-1, <0 → 跳过
              GPU 2: 0-2=-2, <0 → 跳过
              GPU 3: 0-3=-3, <0 → 跳过

            v_id=1:
              GPU 0: 1-0=1, 检查 mb1
              GPU 1: 1-1=0, 检查 mb0 → 可能同步
              GPU 2: 1-2=-1, 跳过
              GPU 3: 1-3=-2, 跳过

            v_id=2:
              GPU 0: 2-0=2, 检查 mb2
              GPU 1: 2-1=1, 检查 mb1
              GPU 2: 2-2=0, 检查 mb0 → 可能同步
              GPU 3: 2-3=-1, 跳过

            v_id=3:
              GPU 0: 3-0=3, 检查 mb3
              GPU 1: 3-1=2, 检查 mb2
              GPU 2: 3-2=1, 检查 mb1
              GPU 3: 3-3=0, 检查 mb0 → 可能同步

            效果：同步操作分散在不同 v_id，减少同时通信竞争
            '''
            grad_sync_virtual_microbatch_id = virtual_microbatch_id - pipeline_parallel_rank
            
            '''
            判断是否是最后一个微批次
            
            grad_sync_v_id >= 0	偏移后的 v_id 有效（不能为负）
            is_last_microbatch_for_model_chunk(...)	该偏移后的 v_id 对应某个 chunk 的最后一个 micro-batch

            为什么是"某个 chunk"而不是"当前 chunk"？
            
            因为错峰策略，不同 rank 在不同 v_id 检查不同 chunk 的完成状态

              GPU 0 在 v_id=3 时检查 chunk X 的 mb3
              GPU 3 在 v_id=3 时检查 chunk Y 的 mb0

              只要某个 chunk 的所有 micro-batch 都反向完成，就可以同步该 chunk 的梯度
            '''
            if grad_sync_virtual_microbatch_id >= 0 and is_last_microbatch_for_model_chunk(
                grad_sync_virtual_microbatch_id
            ):
                '''
                获取同步的 chunk ID
                
                查反向调度表，确定是哪个 chunk 的最后一个 micro-batch
                '''
                grad_sync_chunk_id = get_model_chunk_id(
                    grad_sync_virtual_microbatch_id, forward=False
                )
                
                '''
                执行梯度同步:
                
                enable_grad_sync()	退出 no_sync 模式，允许梯度 All-Reduce
                config.grad_sync_func[chunk_id](...)	执行自定义同步函数（如 ZeRO-3 的 reduce-scatter）
                synchronized_model_chunks.add(...)	记录该 chunk 已同步，避免重复
                '''
                enable_grad_sync()
                config.grad_sync_func[grad_sync_chunk_id](model[grad_sync_chunk_id].parameters())
                synchronized_model_chunks.add(grad_sync_chunk_id)
        
        '''
        禁用梯度同步:
        
        关键操作：无论是否执行了同步，最后都禁用梯度同步
        
        为什么？
        
        下一个 micro-batch 的反向传播：
          - 需要重新进入 no_sync 模式
          - 梯度只累加，不立即 All-Reduce
          - 直到下一个"最后一个 micro-batch"才再次同步

        形成节奏：
          [no_sync] → [反向多个 mb] → [sync] → [no_sync] → [反向多个 mb] → [sync]
        '''
        disable_grad_sync()


#### 与 preprocess 的对比

|          | `backward_step_helper_preprocess` | `backward_step_helper_postprocess`  |
| -------- | --------------------------------- | ----------------------------------- |
| **触发条件** | `config.grad_sync_func is None`   | `config.grad_sync_func is not None` |
| **同步时机** | 默认 DDP 行为                         | 自定义同步，考虑 PP rank 偏移                 |
| **同步对象** | 当前 chunk                          | 通过偏移计算出的 chunk                      |
| **错峰策略** | 无                                 | 有（`v_id - rank`）                    |
| **最后操作** | 无                                 | `disable_grad_sync()`               |


# backward_step
- https://github.com/NVIDIA/Megatron-LM/blob/core_v0.17.0/megatron/core/pipeline_parallel/schedules.py#L451
- 这段代码是 Megatron-LM 中 VPP（虚拟流水线并行）的反向传播核心计算函数，负责执行 PyTorch 的链式法则反向传播并提取输入梯度

| 参数                   | 类型                         | 含义                   |
| -------------------- | -------------------------- | -------------------- |
| `input_tensor`       | `Tensor/List[Tensor]`      | 前向传播的输入激活值           |
| `output_tensor`      | `Tensor/List[Tensor]`      | 前向传播的输出激活值           |
| `output_tensor_grad` | `Tensor/List[Tensor]/None` | 损失对输出的梯度（来自下游 stage） |
| `config`             | `ModelParallelConfig`      | 全局配置                 |


In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              backward_step                                   │
│              （反向传播核心计算）                             │
├─────────────────────────────────────────────────────────────┤
│  1. 输入输出包装（统一列表格式）                              │
│  2. 保留输入梯度（retain_grad）                               │
│  3. 梯度缩放（可选）                                          │
│  4. 执行反向传播（custom_backward 或标准 backward）           │
│  5. 提取输入梯度                                              │
│  6. 解包装返回                                                │
└─────────────────────────────────────────────────────────────┘

In [ ]:

def backward_step(input_tensor, output_tensor, output_tensor_grad, config):
    """Backward step through passed-in output tensor.

    If last stage, output_tensor_grad is None, otherwise gradient of loss
    with respect to stage's output tensor.

    Returns gradient of loss with respect to input tensor (None if first stage)."""

    # NOTE: This code currently can handle at most one skip connection. It
    # needs to be modified slightly to support arbitrary numbers of skip
    # connections.

    if config.timers is not None:
        config.timers('backward-compute', log_level=2).start()

    # Retain the grad on the input_tensor.
    unwrap_input_tensor_grad = False
    if not isinstance(input_tensor, list):
        input_tensor = [input_tensor]
        unwrap_input_tensor_grad = True
    for x in input_tensor:
        if x is not None:
            x.retain_grad()

    if not isinstance(output_tensor, list):
        output_tensor = [output_tensor]
    if not isinstance(output_tensor_grad, list):
        output_tensor_grad = [output_tensor_grad]

    # Backward pass.
    if output_tensor_grad[0] is None and config.grad_scale_func is not None:
        output_tensor[0] = config.grad_scale_func(output_tensor[0])

    # In multi-modal models like VLM, some batches may not have images.
    # When no image is present, the vision encoder (as a separate pipeline stage)
    # will not participate in the computation.
    # This results in a tensor that does not require gradients.
    # In such cases, we intentionally skip the backward pass while preserving zero gradients.
    if output_tensor[0].requires_grad:
        if config.deallocate_pipeline_outputs:
            custom_backward(output_tensor[0], output_tensor_grad[0])
        else:
            torch.autograd.backward(output_tensor[0], grad_tensors=output_tensor_grad[0])

    # Collect the grad of the input_tensor.
    input_tensor_grad = [None]
    if input_tensor is not None:
        input_tensor_grad = []
        for x in input_tensor:
            if x is None:
                input_tensor_grad.append(None)
            else:
                input_tensor_grad.append(x.grad)

    if unwrap_input_tensor_grad:
        input_tensor_grad = input_tensor_grad[0]

    if config.timers is not None:
        config.timers('backward-compute').stop()

    return input_tensor_grad

